# §0.3 KLダイバージェンスとFisher情報 - 情報幾何の核心

## 1. 概要

- **この節で学ぶこと**: KLダイバージェンスの幾何学的意味、Fisher情報行列がリーマン計量になる理由、自然勾配法への接続
- **前提知識**: KLダイバージェンスの定義、Fisher情報行列の計算（あなたの研究経験）
- **情報幾何との関連**: **KLの2次近似がFisher計量を定義する**（最重要の結果）

## 2. 直感的理解

### KLダイバージェンスは「驚きの期待値」

$$D_{\text{KL}}(p \| q) = \mathbb{E}_p\left[ \log \frac{p(x)}{q(x)} \right]$$

- 「$p$の世界に住んでいる人が、$q$だと思ってデータを見たときの驚き」
- 非対称：$p$から見た$q$と、$q$から見た$p$は違う

### 2次近似すると対称になる

近い分布間では：
$$D_{\text{KL}}(p_\theta \| p_{\theta+d\theta}) \approx \frac{1}{2} d\theta^\top I(\theta) d\theta$$

- これは**二次形式**（対称、正定値）
- **Fisher情報行列 $I(\theta)$ がリーマン計量になる**

### あなたの経験との接続

- **情報利得の計算**: $D_{\text{KL}}(\text{posterior} \| \text{prior})$ は観測で得た情報量
- **カルマンフィルタ**: 高精度観測（小さいσ）で更新が大きいのは、Fisher距離が大きいから

## 3. 数学的定義

### 3.1 KLダイバージェンス

**定義**:
$$D_{\text{KL}}(p \| q) = \int p(x) \log \frac{p(x)}{q(x)} dx$$

**性質**:
- 非負: $D_{\text{KL}}(p \| q) \geq 0$
- 非対称: $D_{\text{KL}}(p \| q) \neq D_{\text{KL}}(q \| p)$
- $D_{\text{KL}}(p \| q) = 0 \Leftrightarrow p = q$

### 3.2 Fisher情報行列

**3つの等価な定義**:

1. スコア関数の共分散:
$$I(\theta)_{ij} = \mathbb{E}\left[ \frac{\partial \log p}{\partial \theta_i} \frac{\partial \log p}{\partial \theta_j} \right]$$

2. 対数尤度のヘッセ行列の負の期待値:
$$I(\theta)_{ij} = -\mathbb{E}\left[ \frac{\partial^2 \log p}{\partial \theta_i \partial \theta_j} \right]$$

3. **KLダイバージェンスの2次係数**:
$$D_{\text{KL}}(p_\theta \| p_{\theta+d\theta}) = \frac{1}{2} d\theta^\top I(\theta) d\theta + O(d\theta^3)$$

### 3.3 正規分布のFisher情報行列

$$I(\mu, \sigma) = \begin{pmatrix} 1/\sigma^2 & 0 \\ 0 & 2/\sigma^2 \end{pmatrix}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)

def kl_divergence_gaussian(mu1, sigma1, mu2, sigma2):
    """D_KL(N(μ1,σ1²) || N(μ2,σ2²))"""
    return np.log(sigma2/sigma1) + (sigma1**2 + (mu1-mu2)**2)/(2*sigma2**2) - 0.5

def fisher_matrix_gaussian(sigma):
    """正規分布のFisher情報行列"""
    return np.array([[1/sigma**2, 0], [0, 2/sigma**2]])

## 4. 可視化

### 4.1 KLダイバージェンスの非対称性

In [ ]:
def visualize_kl_asymmetry():
    """KLダイバージェンスの非対称性を可視化"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    mu_p, sigma_p = 0, 1
    mu_q, sigma_q = 2, 1.5
    
    x = np.linspace(-5, 7, 200)
    p = stats.norm.pdf(x, mu_p, sigma_p)
    q = stats.norm.pdf(x, mu_q, sigma_q)
    
    # 左図：分布の比較
    ax1 = axes[0]
    ax1.plot(x, p, 'b-', linewidth=2, label=f'p = N({mu_p}, {sigma_p}²)')
    ax1.plot(x, q, 'r-', linewidth=2, label=f'q = N({mu_q}, {sigma_q}²)')
    ax1.fill_between(x, p, alpha=0.3, color='blue')
    ax1.fill_between(x, q, alpha=0.3, color='red')
    ax1.set_xlabel('x')
    ax1.set_ylabel('Density')
    ax1.set_title('Two Gaussian distributions')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右図：KLの非対称性
    ax2 = axes[1]
    kl_pq = kl_divergence_gaussian(mu_p, sigma_p, mu_q, sigma_q)
    kl_qp = kl_divergence_gaussian(mu_q, sigma_q, mu_p, sigma_p)
    
    bars = ax2.bar(['D_KL(p||q)', 'D_KL(q||p)'], [kl_pq, kl_qp], color=['blue', 'red'])
    ax2.set_ylabel('KL Divergence (nats)')
    ax2.set_title(f'KL Asymmetry\nD_KL(p||q)={kl_pq:.3f}, D_KL(q||p)={kl_qp:.3f}')
    ax2.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, [kl_pq, kl_qp]):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                 f'{val:.3f}', ha='center', fontsize=12)
    
    plt.tight_layout()
    plt.show()

visualize_kl_asymmetry()

### 4.2 KLの2次近似とFisher計量

In [ ]:
def visualize_kl_quadratic_approximation():
    """KLダイバージェンスと2次近似（Fisher計量）の比較"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    mu0, sigma0 = 0, 1
    
    # 左図：μ方向の断面
    ax1 = axes[0]
    dmu_range = np.linspace(-2, 2, 100)
    
    kl_true = [kl_divergence_gaussian(mu0, sigma0, mu0 + dmu, sigma0) for dmu in dmu_range]
    kl_approx = 0.5 * dmu_range**2 / sigma0**2  # Fisher近似
    
    ax1.plot(dmu_range, kl_true, 'b-', linewidth=2, label='True KL')
    ax1.plot(dmu_range, kl_approx, 'r--', linewidth=2, label='Fisher approx')
    ax1.set_xlabel('dμ')
    ax1.set_ylabel('D_KL')
    ax1.set_title('KL along μ direction\nApproximation is exact for Gaussian!')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右図：2D等高線
    ax2 = axes[1]
    dmu = np.linspace(-1, 1, 50)
    dsigma = np.linspace(-0.4, 0.4, 50)
    DMU, DSIGMA = np.meshgrid(dmu, dsigma)
    
    # 真のKL
    KL_TRUE = np.zeros_like(DMU)
    for i in range(len(dmu)):
        for j in range(len(dsigma)):
            if sigma0 + dsigma[j] > 0.1:
                KL_TRUE[j, i] = kl_divergence_gaussian(mu0, sigma0, 
                                                       mu0 + dmu[i], sigma0 + dsigma[j])
    
    # Fisher近似
    KL_APPROX = 0.5 * (DMU**2 / sigma0**2 + 2 * DSIGMA**2 / sigma0**2)
    
    levels = [0.05, 0.1, 0.2, 0.5]
    cs1 = ax2.contour(DMU, DSIGMA, KL_TRUE, levels=levels, colors='blue', linestyles='-')
    cs2 = ax2.contour(DMU, DSIGMA, KL_APPROX, levels=levels, colors='red', linestyles='--')
    ax2.clabel(cs1, inline=True, fontsize=8)
    
    ax2.plot(0, 0, 'ko', markersize=10)
    ax2.set_xlabel('dμ')
    ax2.set_ylabel('dσ')
    ax2.set_title('KL contours: True (blue) vs Fisher (red dashed)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

visualize_kl_quadratic_approximation()

### 4.3 Fisher計量楕円の可視化

In [ ]:
def visualize_fisher_metric():
    """パラメータ空間の各点でFisher計量楕円を描画"""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    theta = np.linspace(0, 2*np.pi, 100)
    
    for mu in np.linspace(-1.5, 1.5, 4):
        for sigma in [0.5, 1.0, 1.5, 2.0]:
            # Fisher計量楕円: dμ²/σ² + 2dσ²/σ² = const
            scale = 0.12
            ellipse_x = sigma * np.cos(theta) * scale + mu
            ellipse_y = sigma / np.sqrt(2) * np.sin(theta) * scale + sigma
            
            color = plt.cm.viridis(sigma / 2.5)
            ax.plot(ellipse_x, ellipse_y, color=color, linewidth=1.5)
            ax.plot(mu, sigma, 'o', color=color, markersize=5)
    
    ax.set_xlabel('μ', fontsize=12)
    ax.set_ylabel('σ', fontsize=12)
    ax.set_title('Fisher metric ellipses\nSmaller σ → smaller ellipse (farther in Fisher distance)', fontsize=11)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(0, 2.5)
    ax.grid(True, alpha=0.3)
    
    # カラーバー的な説明
    ax.annotate('Small σ:\nHigh Fisher info\n(small ellipse)', xy=(-1.5, 0.5), 
                fontsize=10, color='darkblue')
    ax.annotate('Large σ:\nLow Fisher info\n(large ellipse)', xy=(0.5, 2.0), 
                fontsize=10, color='darkgreen')
    
    plt.show()

visualize_fisher_metric()

## 5. 具体例

### 例1：KLの2次近似の導出（正規分布）

In [ ]:
print("""
【KLの2次近似の導出】

D_KL(N(μ₀,σ₀²) || N(μ₀+dμ, (σ₀+dσ)²))
= log((σ₀+dσ)/σ₀) + (σ₀² + dμ²)/(2(σ₀+dσ)²) - 1/2

Taylor展開（dμ, dσについて2次まで）:

log((σ₀+dσ)/σ₀) ≈ dσ/σ₀ - dσ²/(2σ₀²)

1/(σ₀+dσ)² ≈ 1/σ₀² · (1 - 2dσ/σ₀ + 3dσ²/σ₀²)

代入して整理:
D_KL ≈ (1/2) · (dμ²/σ₀² + 2dσ²/σ₀²)
     = (1/2) · [dμ, dσ] · [[1/σ₀², 0], [0, 2/σ₀²]] · [dμ, dσ]ᵀ
     = (1/2) · dθᵀ I(θ) dθ

これがFisher情報行列を導く！
""")

### 例2：情報利得の計算（あなたの研究経験との接続）

In [ ]:
def calculate_information_gain():
    """ベイズ更新による情報利得の計算"""
    # 事前分布
    mu_prior, tau_prior = 0, 2
    
    # 観測（既知のσ=1で）
    sigma_obs = 1.0
    x_obs = 1.5
    
    # 事後分布（正規-正規共役）
    precision_prior = 1 / tau_prior**2
    precision_likelihood = 1 / sigma_obs**2
    precision_post = precision_prior + precision_likelihood
    mu_post = (precision_prior * mu_prior + precision_likelihood * x_obs) / precision_post
    tau_post = 1 / np.sqrt(precision_post)
    
    # 情報利得 = D_KL(posterior || prior)
    info_gain = kl_divergence_gaussian(mu_post, tau_post, mu_prior, tau_prior)
    
    print(f"事前分布: N({mu_prior}, {tau_prior}²)")
    print(f"観測: x = {x_obs} (観測ノイズ σ = {sigma_obs})")
    print(f"事後分布: N({mu_post:.3f}, {tau_post:.3f}²)")
    print(f"\n情報利得 D_KL(posterior || prior) = {info_gain:.4f} nats")
    print(f"\n解釈: 観測により {info_gain:.4f} nats の情報を獲得")

calculate_information_gain()

### 例3：自然勾配の計算

In [ ]:
def compare_gradients():
    """通常勾配と自然勾配の比較"""
    sigma = 0.5  # 小さいσ（高精度領域）
    
    # 通常の勾配（例：損失関数の勾配）
    grad = np.array([1.0, 1.0])
    
    # Fisher情報行列
    I = fisher_matrix_gaussian(sigma)
    I_inv = np.linalg.inv(I)
    
    # 自然勾配
    natural_grad = I_inv @ grad
    
    print(f"σ = {sigma} での計算:")
    print(f"\nFisher情報行列 I:")
    print(I)
    print(f"\n通常勾配: {grad}")
    print(f"自然勾配 I⁻¹∇L: {natural_grad}")
    print(f"\n解釈:")
    print(f"  - μ方向: {grad[0]} → {natural_grad[0]} (σ²={sigma**2}倍に拡大)")
    print(f"  - σ方向: {grad[1]} → {natural_grad[1]} (σ²/2={sigma**2/2}倍に拡大)")
    print(f"\nFisher情報が大きい（精度が高い）ので、自然勾配は小さくなる")

compare_gradients()

## 6. 他の概念との関係

### 前の節との繋がり
- **線形代数 (§0.1)**: Fisher情報行列は正定値行列、自然勾配は双対変換
- **確率・統計 (§0.2)**: 指数型分布族でFisher = 対数分配関数のヘッセ

### 次の節（教科書本論）への接続
- **多様体 (§1.1)**: パラメータ空間が多様体、Fisher計量でリーマン多様体に
- **接ベクトル (§1.2)**: スコア関数が接ベクトルの関数表現
- **リーマン計量 (§1.5)**: Fisher情報行列がまさにリーマン計量

### 情報幾何の核心的関係

$$\boxed{D_{\text{KL}}(p_\theta \| p_{\theta+d\theta}) \approx \frac{1}{2} d\theta^\top I(\theta) d\theta}$$

**この式が情報幾何の出発点！**

## 7. 演習問題

### Q1. KLダイバージェンスの計算

$D_{\text{KL}}(N(0,1) \| N(1,1))$ と $D_{\text{KL}}(N(1,1) \| N(0,1))$ を計算せよ。

<details>
<summary>解答を見る</summary>

$$D_{\text{KL}}(N(0,1) \| N(1,1)) = \log(1) + \frac{1 + 1}{2} - \frac{1}{2} = 0.5$$

$$D_{\text{KL}}(N(1,1) \| N(0,1)) = \log(1) + \frac{1 + 1}{2} - \frac{1}{2} = 0.5$$

σが同じなので対称になる。

</details>

In [ ]:
# Q1検証
kl_01 = kl_divergence_gaussian(0, 1, 1, 1)
kl_10 = kl_divergence_gaussian(1, 1, 0, 1)
print(f"D_KL(N(0,1)||N(1,1)) = {kl_01:.4f}")
print(f"D_KL(N(1,1)||N(0,1)) = {kl_10:.4f}")

### Q2. Fisher情報の計算

ベルヌーイ分布 $\text{Ber}(p)$ のFisher情報 $I(p)$ を計算せよ。

<details>
<summary>解答を見る</summary>

スコア関数: $\frac{\partial \log P(x|p)}{\partial p} = \frac{x-p}{p(1-p)}$

Fisher情報:
$$I(p) = \mathbb{E}\left[\left(\frac{x-p}{p(1-p)}\right)^2\right] = \frac{\text{Var}(x)}{p^2(1-p)^2} = \frac{p(1-p)}{p^2(1-p)^2} = \frac{1}{p(1-p)}$$

</details>

### Q3. 自然勾配

正規分布で $\sigma = 2$ のとき、通常勾配 $(1, 1)$ に対応する自然勾配を計算せよ。

<details>
<summary>解答を見る</summary>

$$I = \begin{pmatrix} 1/4 & 0 \\ 0 & 1/2 \end{pmatrix}, \quad I^{-1} = \begin{pmatrix} 4 & 0 \\ 0 & 2 \end{pmatrix}$$

$$\tilde{\nabla}L = I^{-1}\nabla L = \begin{pmatrix} 4 & 0 \\ 0 & 2 \end{pmatrix}\begin{pmatrix} 1 \\ 1 \end{pmatrix} = \begin{pmatrix} 4 \\ 2 \end{pmatrix}$$

</details>

In [ ]:
# Q3検証
sigma = 2
I = fisher_matrix_gaussian(sigma)
grad = np.array([1, 1])
natural_grad = np.linalg.inv(I) @ grad
print(f"I = \n{I}")
print(f"自然勾配 = {natural_grad}")

## 8. 参考：使用したプロンプト

```
KLダイバージェンスの2次近似がFisher情報行列を導くことを、
正規分布を例に手計算で示してください。途中式も省略せずに。
```

```
カルマンフィルタで観測精度が高いほど更新が大きくなる理由を、
Fisher計量の観点から説明してください。
```

```
情報利得 D_KL(posterior || prior) を計算するPythonコードを書いて、
観測精度を変えたときの情報利得の変化を可視化してください。
```

```
自然勾配法と通常の勾配降下法の違いを、
正規分布のパラメータ推定を例に説明してください。
なぜ自然勾配の方が収束が速いのかを直感的に説明してください。
```

---
**次のステップ**: 第1章 `ch01_differential_geometry/sec01_manifold_basics.ipynb` へ